# Two-tank Fast-DeePC

Fixed benchmark: $T=1500$. This notebook times original hard-$\ell_1$ DeePC, full soft-$\ell_2$ CVXPY/OSQP, and C Fast-DeePC, then draws the tank-level comparison.


## 1. Import packages


In [ ]:
import ctypes as ct
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import cvxpy as cp
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except Exception:
    HAS_MATPLOTLIB = False

np.set_printoptions(precision=4, suppress=True)


## 2. Locate the repository and build the C solver


In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
C_SOURCE = ROOT / "src" / "deepc_boxqp_optimized.c"
WRAPPER_SOURCE = ROOT / "src" / "deepc_boxqp_optimized_ctypes_wrapper.c"
DLL_PATH = ROOT / "build" / "deepc_boxqp_optimized.dll"
PYDEEPC_REPO = Path(os.environ.get("PYDEEPC_REPO", ROOT / "external" / "PyDeePC-main"))
PROJECT_DIR = ROOT
TRY_OPENBLAS = False  # Set True to try -lopenblas before the self-contained wrapper.

if PYDEEPC_REPO.exists() and str(PYDEEPC_REPO) not in sys.path:
    sys.path.insert(0, str(PYDEEPC_REPO))

try:
    import cvxpy as cp
    from pydeepc import DeePC as PyDeePC
    from pydeepc.utils import Data
    HAS_PYDEEPC = True
except Exception as exc:
    HAS_PYDEEPC = False
    PYDEEPC_IMPORT_ERROR = exc

if not C_SOURCE.exists():
    raise FileNotFoundError(C_SOURCE)

def compile_deepc_boxqp():
    gcc = shutil.which("gcc")
    cl = shutil.which("cl")
    if gcc:
        if TRY_OPENBLAS:
            optimized_cmd = [
                gcc,
                "-O3",
                "-march=native",
                "-shared",
                "-Wl,--export-all-symbols",
                "-o",
                str(DLL_PATH),
                str(C_SOURCE),
                "-lopenblas",
                "-lm",
            ]
            print("Trying optimized OpenBLAS build:")
            print(" ".join(map(str, optimized_cmd)))
            optimized = subprocess.run(
                optimized_cmd,
                cwd=PROJECT_DIR,
                text=True,
                capture_output=True,
            )
            if optimized.returncode == 0:
                print(optimized.stdout)
                print("Built with OpenBLAS:", DLL_PATH)
                return DLL_PATH

            print("OpenBLAS build failed; falling back to self-contained wrapper.")
            print(optimized.stderr)
        wrapper_code = '\n#include <math.h>\n#include <ctype.h>\n#include "deepc_boxqp_optimized.c"\n\nstatic int is_t(const char *c) {\n    return c && (c[0] == \'t\' || c[0] == \'T\' || c[0] == \'c\' || c[0] == \'C\');\n}\n\nvoid dcopy_(const long *n, const double *x, const long *incx, double *y, const long *incy) {\n    long i;\n    for (i = 0; i < *n; ++i) y[i * (*incy)] = x[i * (*incx)];\n}\n\nvoid daxpy_(const long *n, const double *alpha, const double *x, const long *incx, double *y, const long *incy) {\n    long i;\n    for (i = 0; i < *n; ++i) y[i * (*incy)] += (*alpha) * x[i * (*incx)];\n}\n\nvoid dscal_(const long *n, const double *alpha, double *x, const long *incx) {\n    long i;\n    for (i = 0; i < *n; ++i) x[i * (*incx)] *= *alpha;\n}\n\ndouble ddot_(const long *n, const double *x, const long *incx, const double *y, const long *incy) {\n    long i;\n    double s = 0.0;\n    for (i = 0; i < *n; ++i) s += x[i * (*incx)] * y[i * (*incy)];\n    return s;\n}\n\nvoid dgemv_(const char *trans, const long *m, const long *n, const double *alpha,\n            const double *A, const long *lda, const double *x, const long *incx,\n            const double *beta, double *y, const long *incy) {\n    long i, j;\n    if (!is_t(trans)) {\n        for (i = 0; i < *m; ++i) {\n            double s = 0.0;\n            for (j = 0; j < *n; ++j) s += A[i + j * (*lda)] * x[j * (*incx)];\n            y[i * (*incy)] = (*alpha) * s + (*beta) * y[i * (*incy)];\n        }\n    } else {\n        for (j = 0; j < *n; ++j) {\n            double s = 0.0;\n            for (i = 0; i < *m; ++i) s += A[i + j * (*lda)] * x[i * (*incx)];\n            y[j * (*incy)] = (*alpha) * s + (*beta) * y[j * (*incy)];\n        }\n    }\n}\n\nvoid dgemm_(const char *transa, const char *transb, const long *m, const long *n, const long *k,\n            const double *alpha, const double *A, const long *lda, const double *B, const long *ldb,\n            const double *beta, double *C, const long *ldc) {\n    long i, j, l;\n    int ta = is_t(transa);\n    int tb = is_t(transb);\n    for (j = 0; j < *n; ++j) {\n        for (i = 0; i < *m; ++i) {\n            double s = 0.0;\n            for (l = 0; l < *k; ++l) {\n                double a = ta ? A[l + i * (*lda)] : A[i + l * (*lda)];\n                double b = tb ? B[j + l * (*ldb)] : B[l + j * (*ldb)];\n                s += a * b;\n            }\n            C[i + j * (*ldc)] = (*alpha) * s + (*beta) * C[i + j * (*ldc)];\n        }\n    }\n}\n\nvoid dpotrf_(const char *uplo, const long *n, double *A, const long *lda, long *info) {\n    long i, j, k;\n    *info = 0;\n    if (uplo && (uplo[0] == \'u\' || uplo[0] == \'U\')) {\n        *info = -1;\n        return;\n    }\n    for (j = 0; j < *n; ++j) {\n        double diag = A[j + j * (*lda)];\n        for (k = 0; k < j; ++k) diag -= A[j + k * (*lda)] * A[j + k * (*lda)];\n        if (!(diag > 0.0) || !isfinite(diag)) {\n            *info = j + 1;\n            return;\n        }\n        A[j + j * (*lda)] = sqrt(diag);\n        for (i = j + 1; i < *n; ++i) {\n            double s = A[i + j * (*lda)];\n            for (k = 0; k < j; ++k) s -= A[i + k * (*lda)] * A[j + k * (*lda)];\n            A[i + j * (*lda)] = s / A[j + j * (*lda)];\n        }\n        for (k = j + 1; k < *n; ++k) A[j + k * (*lda)] = 0.0;\n    }\n}\n\nvoid dpotrs_(const char *uplo, const long *n, const long *nrhs, const double *A, const long *lda,\n             double *B, const long *ldb, long *info) {\n    long i, j, k;\n    *info = 0;\n    if (uplo && (uplo[0] == \'u\' || uplo[0] == \'U\')) {\n        *info = -1;\n        return;\n    }\n    for (j = 0; j < *nrhs; ++j) {\n        for (i = 0; i < *n; ++i) {\n            double s = B[i + j * (*ldb)];\n            for (k = 0; k < i; ++k) s -= A[i + k * (*lda)] * B[k + j * (*ldb)];\n            B[i + j * (*ldb)] = s / A[i + i * (*lda)];\n        }\n        for (i = *n - 1; i >= 0; --i) {\n            double s = B[i + j * (*ldb)];\n            for (k = i + 1; k < *n; ++k) s -= A[k + i * (*lda)] * B[k + j * (*ldb)];\n            B[i + j * (*ldb)] = s / A[i + i * (*lda)];\n        }\n    }\n}\n'
        WRAPPER_SOURCE.write_text(wrapper_code, encoding="utf-8")
        cmd = [
            gcc,
            "-O2",
            "-shared",
            "-Wl,--export-all-symbols",
            "-o",
            str(DLL_PATH),
            str(WRAPPER_SOURCE),
            "-lm",
        ]
    elif cl:
        wrapper_code = '\n#include <math.h>\n#include <ctype.h>\n#include "deepc_boxqp_optimized.c"\n\nstatic int is_t(const char *c) {\n    return c && (c[0] == \'t\' || c[0] == \'T\' || c[0] == \'c\' || c[0] == \'C\');\n}\n\nvoid dcopy_(const long *n, const double *x, const long *incx, double *y, const long *incy) {\n    long i;\n    for (i = 0; i < *n; ++i) y[i * (*incy)] = x[i * (*incx)];\n}\n\nvoid daxpy_(const long *n, const double *alpha, const double *x, const long *incx, double *y, const long *incy) {\n    long i;\n    for (i = 0; i < *n; ++i) y[i * (*incy)] += (*alpha) * x[i * (*incx)];\n}\n\nvoid dscal_(const long *n, const double *alpha, double *x, const long *incx) {\n    long i;\n    for (i = 0; i < *n; ++i) x[i * (*incx)] *= *alpha;\n}\n\ndouble ddot_(const long *n, const double *x, const long *incx, const double *y, const long *incy) {\n    long i;\n    double s = 0.0;\n    for (i = 0; i < *n; ++i) s += x[i * (*incx)] * y[i * (*incy)];\n    return s;\n}\n\nvoid dgemv_(const char *trans, const long *m, const long *n, const double *alpha,\n            const double *A, const long *lda, const double *x, const long *incx,\n            const double *beta, double *y, const long *incy) {\n    long i, j;\n    if (!is_t(trans)) {\n        for (i = 0; i < *m; ++i) {\n            double s = 0.0;\n            for (j = 0; j < *n; ++j) s += A[i + j * (*lda)] * x[j * (*incx)];\n            y[i * (*incy)] = (*alpha) * s + (*beta) * y[i * (*incy)];\n        }\n    } else {\n        for (j = 0; j < *n; ++j) {\n            double s = 0.0;\n            for (i = 0; i < *m; ++i) s += A[i + j * (*lda)] * x[i * (*incx)];\n            y[j * (*incy)] = (*alpha) * s + (*beta) * y[j * (*incy)];\n        }\n    }\n}\n\nvoid dgemm_(const char *transa, const char *transb, const long *m, const long *n, const long *k,\n            const double *alpha, const double *A, const long *lda, const double *B, const long *ldb,\n            const double *beta, double *C, const long *ldc) {\n    long i, j, l;\n    int ta = is_t(transa);\n    int tb = is_t(transb);\n    for (j = 0; j < *n; ++j) {\n        for (i = 0; i < *m; ++i) {\n            double s = 0.0;\n            for (l = 0; l < *k; ++l) {\n                double a = ta ? A[l + i * (*lda)] : A[i + l * (*lda)];\n                double b = tb ? B[j + l * (*ldb)] : B[l + j * (*ldb)];\n                s += a * b;\n            }\n            C[i + j * (*ldc)] = (*alpha) * s + (*beta) * C[i + j * (*ldc)];\n        }\n    }\n}\n\nvoid dpotrf_(const char *uplo, const long *n, double *A, const long *lda, long *info) {\n    long i, j, k;\n    *info = 0;\n    if (uplo && (uplo[0] == \'u\' || uplo[0] == \'U\')) {\n        *info = -1;\n        return;\n    }\n    for (j = 0; j < *n; ++j) {\n        double diag = A[j + j * (*lda)];\n        for (k = 0; k < j; ++k) diag -= A[j + k * (*lda)] * A[j + k * (*lda)];\n        if (!(diag > 0.0) || !isfinite(diag)) {\n            *info = j + 1;\n            return;\n        }\n        A[j + j * (*lda)] = sqrt(diag);\n        for (i = j + 1; i < *n; ++i) {\n            double s = A[i + j * (*lda)];\n            for (k = 0; k < j; ++k) s -= A[i + k * (*lda)] * A[j + k * (*lda)];\n            A[i + j * (*lda)] = s / A[j + j * (*lda)];\n        }\n        for (k = j + 1; k < *n; ++k) A[j + k * (*lda)] = 0.0;\n    }\n}\n\nvoid dpotrs_(const char *uplo, const long *n, const long *nrhs, const double *A, const long *lda,\n             double *B, const long *ldb, long *info) {\n    long i, j, k;\n    *info = 0;\n    if (uplo && (uplo[0] == \'u\' || uplo[0] == \'U\')) {\n        *info = -1;\n        return;\n    }\n    for (j = 0; j < *nrhs; ++j) {\n        for (i = 0; i < *n; ++i) {\n            double s = B[i + j * (*ldb)];\n            for (k = 0; k < i; ++k) s -= A[i + k * (*lda)] * B[k + j * (*ldb)];\n            B[i + j * (*ldb)] = s / A[i + i * (*lda)];\n        }\n        for (i = *n - 1; i >= 0; --i) {\n            double s = B[i + j * (*ldb)];\n            for (k = i + 1; k < *n; ++k) s -= A[k + i * (*lda)] * B[k + j * (*ldb)];\n            B[i + j * (*ldb)] = s / A[i + i * (*lda)];\n        }\n    }\n}\n'
        WRAPPER_SOURCE.write_text(wrapper_code, encoding="utf-8")
        cmd = [cl, "/O2", "/LD", str(WRAPPER_SOURCE), f"/Fe:{DLL_PATH}"]
    else:
        raise RuntimeError(
            "No C compiler found. Install MSYS2/UCRT64 gcc or run this from a Visual Studio Developer shell."
        )

    print("Compiling with:")
    print(" ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"compile failed with exit code {result.returncode}")
    return DLL_PATH


dll_path = compile_deepc_boxqp()
print("Built:", dll_path)


## 3. Load the C Fast-DeePC interface


In [ ]:
BINT = ct.c_long


class BoxQPInfo(ct.Structure):
    _fields_ = [
        ("iterations", BINT),
        ("mu", ct.c_double),
        ("duality", ct.c_double),
        ("status", ct.c_int),
        ("chol_info", BINT),
    ]


class CDeePC(ct.Structure):
    _fields_ = [
        ("m", BINT), ("p", BINT), ("Tini", BINT), ("N", BINT),
        ("L", BINT), ("ncol", BINT), ("nz", BINT), ("nu", BINT), ("ny", BINT),
        ("Up", ct.POINTER(ct.c_double)),
        ("Yp", ct.POINTER(ct.c_double)),
        ("Uf", ct.POINTER(ct.c_double)),
        ("Yf", ct.POINTER(ct.c_double)),
        ("Z", ct.POINTER(ct.c_double)),
        ("nZrows", BINT),
        ("Mchol", ct.POINTER(ct.c_double)),
        ("Hhat", ct.POINTER(ct.c_double)),
        ("Wu", ct.POINTER(ct.c_double)),
        ("Wy", ct.POINTER(ct.c_double)),
        ("lambda_", ct.c_double),
        ("rho", ct.c_double),
        ("lambda_g", ct.c_double),
        ("qt", ct.POINTER(ct.c_double)),
        ("ct", ct.POINTER(ct.c_double)),
        ("wini", ct.POINTER(ct.c_double)),
        ("hhat", ct.POINTER(ct.c_double)),
    ]


if hasattr(os, "add_dll_directory"):
    for dll_dir in [DLL_PATH.parent, Path(r"C:\msys64\ucrt64\bin")]:
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))

lib = ct.CDLL(str(DLL_PATH))
double_p = ct.POINTER(ct.c_double)

lib.deepc_offline.argtypes = [
    ct.POINTER(CDeePC),
    double_p, double_p, BINT,
    BINT, BINT, BINT, BINT,
    double_p, double_p,
    ct.c_double, ct.c_double, ct.c_double,
]
lib.deepc_offline.restype = ct.c_int

lib.deepc_online_step.argtypes = [
    ct.POINTER(CDeePC),
    double_p, double_p, double_p,
    double_p,
    ct.c_double, BINT,
    double_p,
    ct.POINTER(BoxQPInfo),
]
lib.deepc_online_step.restype = ct.c_int

lib.deepc_free.argtypes = [ct.POINTER(CDeePC)]
lib.deepc_free.restype = None


def ptr(a):
    return np.asarray(a, dtype=np.float64).ctypes.data_as(double_p)


def fmat(a):
    return np.asfortranarray(a, dtype=np.float64)


print("Loaded:", DLL_PATH)


## 4. Define the fixed two-tank system and DeePC helpers


In [ ]:
s = 1
T_INI = 2
HORIZON = 50
EXPERIMENT_HORIZON = 30

T_DATA = 1500
LAMBDA_PEN = 1e6
RHO_PEN = 1e6
LAMBDA_G_REGULARIZER = 2.0

WY_TRACK = 2.0
WU_TRACK = 0.0

Y_LOWER = -1.0
Y_UPPER = 1.0

SEED = 4
noise_std = 0.0

A = np.array([
    [0.70469, 0.0],
    [0.24664, 0.70469],
], dtype=np.float64)
B = np.array([
    [0.75937, 0.75937],
    [0.12515, 0.0],
], dtype=np.float64)
C = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
], dtype=np.float64)
D = np.zeros((C.shape[0], B.shape[1]), dtype=np.float64)

dim_x, dim_u = B.shape
dim_y = C.shape[0]
nu = HORIZON * dim_u
ny = HORIZON * dim_y
nz = nu + ny


def simulate_lti(U, noise=None, x0=None):
    U = np.asarray(U, dtype=np.float64)
    x = np.zeros(dim_x, dtype=np.float64) if x0 is None else np.asarray(x0, dtype=np.float64).copy()
    Y = np.zeros((len(U), dim_y), dtype=np.float64)
    X = np.zeros((len(U) + 1, dim_x), dtype=np.float64)
    X[0] = x
    if noise is None:
        noise = np.zeros((len(U), dim_y), dtype=np.float64)
    for k, u in enumerate(U):
        Y[k] = C @ x + D @ u + noise[k]
        x = A @ x + B @ u
        X[k + 1] = x
    return Y, X


def make_shared_sequences(T, seed):
    rng = np.random.default_rng(seed)
    u_data = rng.normal(size=(T, dim_u)).astype(np.float64)
    offline_noise = rng.normal(scale=noise_std, size=(T, dim_y)).astype(np.float64)
    online_noise = rng.normal(scale=noise_std, size=(EXPERIMENT_HORIZON, dim_y)).astype(np.float64)
    y_data, _ = simulate_lti(u_data, noise=offline_noise)
    return u_data, y_data, online_noise


def update_window(u_window, y_window, u_t, y_t):
    return (
        np.vstack([u_window[1:], u_t.reshape(1, -1)]),
        np.vstack([y_window[1:], y_t.reshape(1, -1)]),
    )

def make_hankel(data, order):
    data = np.asarray(data, dtype=np.float64)
    T, channels = data.shape
    ncol = T - order + 1
    H = np.zeros((order * channels, ncol), dtype=np.float64)
    for j in range(ncol):
        H[:, j] = data[j:j + order, :].reshape(-1)
    return H


def split_hankel(u_data, y_data):
    L = T_INI + HORIZON
    Hu = make_hankel(u_data, L)
    Hy = make_hankel(y_data, L)
    Up = Hu[:T_INI * dim_u, :]
    Uf = Hu[T_INI * dim_u:, :]
    Yp = Hy[:T_INI * dim_y, :]
    Yf = Hy[T_INI * dim_y:, :]
    return Up, Yp, Uf, Yf


def solve_cvxpy_quadratic_step(u_data, y_data, uini, yini, reference):
    Up, Yp, Uf, Yf = split_hankel(u_data, y_data)
    ncol = Uf.shape[1]
    g = cp.Variable(ncol)
    u = cp.Variable(nu)
    y = cp.Variable(ny)

    Wu_diag = WU_TRACK * np.ones(nu)
    Wy_diag = WY_TRACK * np.ones(ny)
    r = np.asarray(reference, dtype=np.float64).reshape(-1)
    uini_vec = np.asarray(uini, dtype=np.float64).reshape(-1)
    yini_vec = np.asarray(yini, dtype=np.float64).reshape(-1)

    objective = 0
    if WU_TRACK != 0:
        objective += 0.5 * cp.sum(cp.multiply(Wu_diag, cp.square(u)))
    objective += 0.5 * cp.sum(cp.multiply(Wy_diag, cp.square(y - r)))
    objective += 0.5 * LAMBDA_PEN * cp.sum_squares(Up @ g - uini_vec)
    objective += 0.5 * LAMBDA_PEN * cp.sum_squares(Yp @ g - yini_vec)
    objective += 0.5 * LAMBDA_PEN * cp.sum_squares(Uf @ g - u)
    objective += 0.5 * RHO_PEN * cp.sum_squares(Yf @ g - y)
    objective += 0.5 * LAMBDA_G_REGULARIZER * cp.sum_squares(g)

    constraints = [u >= -1, u <= 1, y >= Y_LOWER, y <= Y_UPPER]
    problem = cp.Problem(cp.Minimize(objective), constraints)

    solvers = [s for s in ["OSQP", "CLARABEL", "SCS"] if s in cp.installed_solvers()]
    last_error = None
    result = None
    for solver_name in solvers:
        try:
            kwargs = {"solver": solver_name, "warm_start": True}
            if solver_name == "OSQP":
                kwargs.update({"eps_abs": 1e-7, "eps_rel": 1e-7, "max_iter": 200000})
            result = problem.solve(**kwargs)
            break
        except Exception as exc:
            last_error = exc
    if result is None:
        raise RuntimeError(f"CVXPY quadratic solve failed. Last error: {last_error}")

    return {
        "u_plan": np.asarray(u.value, dtype=np.float64).reshape(HORIZON, dim_u),
        "y_plan": np.asarray(y.value, dtype=np.float64).reshape(HORIZON, dim_y),
        "g": np.asarray(g.value, dtype=np.float64),
        "objective": float(result),
        "solver": problem.solver_stats.solver_name,
        "solve_time": float(problem.solver_stats.solve_time or 0.0),
        "status": problem.status,
    }


def make_c_controller(u_data, y_data):
    Wu = fmat(WU_TRACK * np.eye(nu))
    Wy = fmat(WY_TRACK * np.eye(ny))
    deepc = CDeePC()
    t0 = time.perf_counter()
    status = lib.deepc_offline(
        ct.byref(deepc),
        ptr(u_data.reshape(-1)),
        ptr(y_data.reshape(-1)),
        BINT(len(u_data)),
        BINT(dim_u), BINT(dim_y), BINT(T_INI), BINT(HORIZON),
        ptr(Wu), ptr(Wy),
        ct.c_double(LAMBDA_PEN),
        ct.c_double(RHO_PEN),
        ct.c_double(LAMBDA_G_REGULARIZER),
    )
    offline_time = time.perf_counter() - t0
    if status != 0:
        raise RuntimeError(f"deepc_offline failed with status {status}")
    return deepc, offline_time


def solve_c_step(deepc, uini, yini, reference, z_warm=None):
    z_out = np.zeros(nz, dtype=np.float64)
    info = BoxQPInfo()
    warm_ptr = None if z_warm is None else ptr(z_warm)
    t0 = time.perf_counter()
    status = lib.deepc_online_step(
        ct.byref(deepc),
        ptr(np.asarray(uini, dtype=np.float64).reshape(-1)),
        ptr(np.asarray(yini, dtype=np.float64).reshape(-1)),
        ptr(np.asarray(reference, dtype=np.float64).reshape(-1)),
        warm_ptr,
        ct.c_double(1e-8),
        BINT(80),
        ptr(z_out),
        ct.byref(info),
    )
    solve_time = time.perf_counter() - t0
    if status not in (0, 1):
        raise RuntimeError(f"deepc_online_step failed status={status}, chol_info={info.chol_info}")
    return {
        "z": z_out,
        "u_plan": z_out[:nu].reshape(HORIZON, dim_u),
        "y_plan": z_out[nu:].reshape(HORIZON, dim_y),
        "solve_time": solve_time,
        "iterations": int(info.iterations),
        "mu": float(info.mu),
        "status": int(status),
    }


## 5. Build the three fixed controllers once


In [ ]:
# Build the three controllers once. The full soft-L2 CVXPY controller
# is parameterized, so repeated solves do not rebuild or canonicalize the problem.
TIMING_REPEATS = 10
TIMING_WARMUPS = 1

def make_reference_schedule(length):
    """Common output reference for both tanks: 0.8, then 0.5, then 1.0."""
    schedule = np.ones((length, dim_y), dtype=np.float64)
    schedule[:10, :] = 0.8
    schedule[10:20, :] = 0.5
    return schedule

reference_schedule = make_reference_schedule(EXPERIMENT_HORIZON + HORIZON)

def reference_for_time(k):
    indices = np.minimum(np.arange(HORIZON) + k, len(reference_schedule) - 1)
    return reference_schedule[indices].reshape(-1)

class FullSoftL2CVXPY:
    def __init__(self, Up, Yp, Uf, Yf):
        ng = Uf.shape[1]
        self.g = cp.Variable(ng)
        self.u = cp.Variable(nu)
        self.y = cp.Variable(ny)
        self.uini = cp.Parameter(T_INI * dim_u)
        self.yini = cp.Parameter(T_INI * dim_y)
        self.reference = cp.Parameter(ny)
        cost = (
            0.5 * WY_TRACK * cp.sum_squares(self.y - self.reference)
            + 0.5 * WU_TRACK * cp.sum_squares(self.u)
            + 0.5 * LAMBDA_PEN * cp.sum_squares(Up @ self.g - self.uini)
            + 0.5 * LAMBDA_PEN * cp.sum_squares(Yp @ self.g - self.yini)
            + 0.5 * LAMBDA_PEN * cp.sum_squares(Uf @ self.g - self.u)
            + 0.5 * RHO_PEN * cp.sum_squares(Yf @ self.g - self.y)
            + 0.5 * LAMBDA_G_REGULARIZER * cp.sum_squares(self.g)
        )
        self.problem = cp.Problem(cp.Minimize(cost), [
            self.u >= -1, self.u <= 1,
            self.y >= Y_LOWER, self.y <= Y_UPPER,
        ])

    def solve(self, uini, yini, reference):
        self.uini.value = np.asarray(uini, dtype=np.float64).reshape(-1)
        self.yini.value = np.asarray(yini, dtype=np.float64).reshape(-1)
        self.reference.value = np.asarray(reference, dtype=np.float64).reshape(-1)
        start = time.perf_counter()
        objective = self.problem.solve(
            solver=cp.OSQP, warm_start=True, eps_abs=1e-7, eps_rel=1e-7,
            max_iter=200000, polish=False, verbose=False,
        )
        elapsed = time.perf_counter() - start
        if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(f"full soft-L2 CVXPY status: {self.problem.status}")
        return {
            "u": np.asarray(self.u.value).reshape(HORIZON, dim_u),
            "y": np.asarray(self.y.value).reshape(HORIZON, dim_y),
            "objective": float(objective), "time": elapsed,
        }


class OriginalHardL1DeePC:
    """Hard-equality/L1 DeePC baseline with an online reference parameter.

    This is the original DeePC optimization formulation written directly in
    CVXPY. It is used instead of the PyDeePC wrapper because PyDeePC's loss
    callback fixes the reference when the problem is built.
    """
    def __init__(self, u_data, y_data):
        Up, Yp, Uf, Yf = split_hankel(u_data, y_data)
        ng = Uf.shape[1]
        self.g = cp.Variable(ng)
        self.u = cp.Variable(nu)
        self.y = cp.Variable(ny)
        self.uini = cp.Parameter(T_INI * dim_u)
        self.yini = cp.Parameter(T_INI * dim_y)
        self.reference = cp.Parameter(ny)
        objective = (
            0.5 * WY_TRACK * cp.sum_squares(self.y - self.reference)
            + 0.5 * WU_TRACK * cp.sum_squares(self.u)
            + LAMBDA_G_REGULARIZER * cp.norm1(self.g)
        )
        self.problem = cp.Problem(cp.Minimize(objective), [
            Up @ self.g == self.uini,
            Yp @ self.g == self.yini,
            Uf @ self.g == self.u,
            Yf @ self.g == self.y,
            self.u >= -1, self.u <= 1,
            self.y >= Y_LOWER, self.y <= Y_UPPER,
        ])

    def solve(self, uini, yini, reference):
        start = time.perf_counter()
        self.uini.value = np.asarray(uini, dtype=np.float64).reshape(-1)
        self.yini.value = np.asarray(yini, dtype=np.float64).reshape(-1)
        self.reference.value = np.asarray(reference, dtype=np.float64).reshape(-1)
        attempts = []
        # CVXPY treats norm1(g) as conic in this installed version, so use a
        # conic solver rather than forcing OSQP.
        if "CLARABEL" in cp.installed_solvers():
            attempts.append(("CLARABEL", {
                "solver": cp.CLARABEL, "warm_start": True,
                "tol_gap_abs": 1e-7, "tol_feas": 1e-7, "max_iter": 10000,
            }))
        if "SCS" in cp.installed_solvers():
            attempts.append(("SCS", {
                "solver": cp.SCS, "warm_start": True,
                "eps": 1e-5, "max_iters": 100000,
            }))
        if not attempts:
            raise RuntimeError("Original hard-L1 DeePC needs CLARABEL or SCS.")

        errors = []
        for solver_name, kwargs in attempts:
            try:
                self.problem.solve(**kwargs)
                if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
                    raise RuntimeError(f"status={self.problem.status}")
                return {
                    "u": np.asarray(self.u.value).reshape(HORIZON, dim_u),
                    "y": np.asarray(self.y.value).reshape(HORIZON, dim_y),
                    "time": time.perf_counter() - start,
                    "solver": solver_name,
                }
            except Exception as exc:
                errors.append(f"{solver_name}: {exc}")
        raise RuntimeError("Original hard-L1 DeePC failed. " + " | ".join(errors))


## 6. Measure ten online solves at $T=1500$`n


In [ ]:
u_data, y_data, _ = make_shared_sequences(T_DATA, seed=SEED + T_DATA)
Up, Yp, Uf, Yf = split_hankel(u_data, y_data)
uini = np.zeros((T_INI, dim_u), dtype=np.float64)
yini = np.zeros((T_INI, dim_y), dtype=np.float64)
reference = reference_for_time(0)

print("Building controllers once (offline work excluded from timing)...")
t0 = time.perf_counter(); hard = OriginalHardL1DeePC(u_data, y_data); hard_offline_s = time.perf_counter() - t0
t0 = time.perf_counter(); soft = FullSoftL2CVXPY(Up, Yp, Uf, Yf); soft_offline_s = time.perf_counter() - t0
t0 = time.perf_counter(); c_deepc, c_offline_s = make_c_controller(u_data, y_data); c_offline_s += time.perf_counter() - t0 - c_offline_s
print(f"g dimension = {Uf.shape[1]}, reduced BoxQP dimension = {nz}")


# Ten repeated online solves. Each method receives exactly the same window and reference.
# One warm-up is excluded to avoid first-call allocation and factorization overhead.
hard.solve(uini, yini, reference)
soft.solve(uini, yini, reference)
c_warm = solve_c_step(c_deepc, uini, yini, reference)["z"]

times = {"Original hard-L1 PyDeePC": [], "Full soft-L2 CVXPY/OSQP": [], "Reduced soft-L2 C BoxQP": []}
for repeat in range(TIMING_REPEATS):
    order = ["hard", "soft", "c"]
    order = order[repeat % 3:] + order[:repeat % 3]
    for method in order:
        if method == "hard":
            times["Original hard-L1 PyDeePC"].append(hard.solve(uini, yini, reference)["time"])
        elif method == "soft":
            times["Full soft-L2 CVXPY/OSQP"].append(soft.solve(uini, yini, reference)["time"])
        else:
            times["Reduced soft-L2 C BoxQP"].append(
                solve_c_step(c_deepc, uini, yini, reference, z_warm=c_warm)["solve_time"]
            )

times_ms = {name: 1e3 * np.asarray(values) for name, values in times.items()}
print(f"T={T_DATA}; {TIMING_REPEATS} timed online solves per controller")
for name, values in times_ms.items():
    print(f"{name:30s} mean = {values.mean():9.3f} ms; std = {values.std(ddof=1):8.3f} ms")

soft_c_speedup = times_ms["Full soft-L2 CVXPY/OSQP"].mean() / times_ms["Reduced soft-L2 C BoxQP"].mean()
hard_c_ratio = times_ms["Original hard-L1 PyDeePC"].mean() / times_ms["Reduced soft-L2 C BoxQP"].mean()
print(f"\nSame-problem speedup, full soft-L2 CVXPY / C BoxQP: {soft_c_speedup:.1f}x")
print(f"Original hard-L1 PyDeePC / C BoxQP timing ratio: {hard_c_ratio:.1f}x")


## 7. Plot the original L1 DeePC and C Fast-DeePC tank-level responses


In [ ]:
# Output-only control comparison. No RMSE is calculated or displayed.
# Both controllers receive the same 0.8 -> 0.5 -> 1.0 reference schedule,
# but retain their distinct hard-L1 and soft-L2 formulations.
def simulate_closed_loop(method):
    x = np.zeros(dim_x, dtype=np.float64)
    uw = np.zeros((T_INI, dim_u), dtype=np.float64)
    yw = np.zeros((T_INI, dim_y), dtype=np.float64)
    out, z_warm = [], None
    for k in range(EXPERIMENT_HORIZON):
        current_reference = reference_for_time(k)
        if method == "hard":
            action = hard.solve(uw, yw, current_reference)["u"][0]
        else:
            answer = solve_c_step(c_deepc, uw, yw, current_reference, z_warm=z_warm)
            action, z_warm = answer["u_plan"][0], answer["z"]
        y = C @ x + D @ action
        x = A @ x + B @ action
        out.append(y)
        uw, yw = update_window(uw, yw, action, y)
    return np.asarray(out)

y_hard = simulate_closed_loop("hard")
y_c = simulate_closed_loop("c")
steps = np.arange(EXPERIMENT_HORIZON)

fig, axes = plt.subplots(2, 1, figsize=(9.1, 5.7), sharex=True)
for j, axis in enumerate(axes):
    axis.plot(steps, y_hard[:, j], color="tab:blue", linewidth=2.4, label="Original hard-L1 DeePC")
    axis.plot(steps, y_c[:, j], color="tab:orange", linestyle="--", linewidth=2.6, label="C Fast-DeePC BoxQP")
    axis.plot(steps, reference_schedule[:EXPERIMENT_HORIZON, j], color="black", linestyle="--", linewidth=1.3, label="reference")
    axis.axhline(Y_LOWER, color="black", linestyle=":", linewidth=1.0, label="output bounds")
    axis.axhline(Y_UPPER, color="black", linestyle=":", linewidth=1.0)
    axis.set_ylabel(rf"tank {j + 1} level $y_{j + 1}$ (normalized)", fontsize=12)
    axis.set_ylim(-1.1, 1.1)
    axis.grid(True, alpha=0.3)
    axis.legend(loc="lower right", fontsize=10)
axes[-1].set_xlabel("closed-loop step (sample)", fontsize=12)
fig.subplots_adjust(left=0.16, right=0.98, bottom=0.12, top=0.98, hspace=0.28)
plt.savefig("final2tank_t1500_outputs_l1_vs_c.png", dpi=300, bbox_inches="tight", pad_inches=0.18)
plt.show()

lib.deepc_free(ct.byref(c_deepc))
print("Saved final2tank_t1500_timing.png and final2tank_t1500_outputs_l1_vs_c.png")
